In [5]:
import os
import shutil
import pandas as pd
from datetime import datetime
from tqdm import tqdm
from concurrent.futures import ThreadPoolExecutor

# ==========================================
# 1. 입력 경로 및 설정 (수정 용이)
# ==========================================
SOURCE_ROOT = '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/251213-251226'  # 원본 이미지 루트 폴더
TARGET_ROOT = '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226'  # 결과 저장 폴더
TOTAL_LIST_PATH = '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226/total_image_list.xlsx'
FILTERED_LIST_PATH = '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226/filtered_image_list.xlsx'

# 파일명 규칙 설정 (예: BED01_20241215_103025.jpg)
# 구분자('_')를 기준으로 인덱스 설정
BED_IDX = 0
DATE_IDX = 1
TIME_IDX = 2


In [6]:


# ==========================================
# 2. 파일 탐색 및 리스트 생성 함수
# ==========================================
def get_all_images(root_dir):
    """하위 폴더를 포함한 모든 이미지 경로 탐색"""
    all_files = []
    print(f"[*] '{root_dir}'에서 파일 탐색 중...")

    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                full_path = os.path.join(root, file)
                rel_path = os.path.relpath(full_path, root_dir)

                # 파일명 파싱
                name_parts = file.replace('.jpg', '').replace('.png', '').split('_')
                if len(name_parts) >= 3:
                    all_files.append({
                        'file_name': file,
                        'rel_path': rel_path,
                        'full_path': full_path,
                        'sub_dir': os.path.basename(root),
                        'bed_id': name_parts[BED_IDX],
                        'date': name_parts[DATE_IDX],
                        'time': name_parts[TIME_IDX]
                    })

    df = pd.DataFrame(all_files)
    df.to_excel(TOTAL_LIST_PATH, index=False)
    print(f"[V] 전체 이미지 리스트 저장 완료: {len(df)}건")
    return df

# ==========================================
# 3. 최초 시간 데이터 필터링 함수
# ==========================================
def filter_earliest_images(df):
    """같은 Bed, 같은 Date 중 가장 빠른 시간만 선택"""
    print("[*] 베드/날짜별 최초 시간 사진 선별 중...")

    # 시간순 정렬 후 첫 번째 값 선택
    filtered_df = df.sort_values(by=['bed_id', 'date', 'time']).groupby(['bed_id', 'date']).first().reset_index()

    filtered_df.to_excel(FILTERED_LIST_PATH, index=False)
    print(f"[V] 필터링 완료: {len(filtered_df)}건 (중복 제거됨)")
    return filtered_df

# ==========================================
# 4. 병렬 파일 복사 함수
# ==========================================
def copy_single_file(row):
    """개별 파일 복사 (하위 구조 유지)"""
    src_path = row['full_path']
    dest_path = os.path.join(TARGET_ROOT, row['rel_path'])

    # 목적지 폴더 생성
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)

    # 복사 (이미 있으면 건너뛰기 - 필요 시 설정)
    if not os.path.exists(dest_path):
        shutil.copy2(src_path, dest_path)
    return True

def parallel_copy(filtered_df):
    """Thread를 사용한 빠른 병렬 복사"""
    print(f"[*] '{TARGET_ROOT}'로 파일 복사 시작 (병렬 작업)...")

    rows = filtered_df.to_dict('records')

    # CPU 코어 수에 맞춰 작업자 지정 (보통 4~8 권장)
    with ThreadPoolExecutor(max_workers=8) as executor:
        list(tqdm(executor.map(copy_single_file, rows), total=len(rows), desc="복사 진행률"))


In [7]:

# ==========================================
# 실행 메인 로직
# ==========================================
if __name__ == "__main__":
    start_time = datetime.now()

    # 1단계: 전체 탐색
    total_df = get_all_images(SOURCE_ROOT)

    if not total_df.empty:
        # 2단계: 필터링
        filtered_df = filter_earliest_images(total_df)

        # 3단계: 병렬 복사
        parallel_copy(filtered_df)
    else:
        print("[!] 탐색된 이미지 파일이 없습니다. 경로를 확인해주세요.")

    end_time = datetime.now()
    print(f"\n[★] 모든 작업이 완료되었습니다.")
    print(f"총 소요 시간: {end_time - start_time}")

[*] '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/0. 원본/251213-251226'에서 파일 탐색 중...
[V] 전체 이미지 리스트 저장 완료: 6961건
[*] 베드/날짜별 최초 시간 사진 선별 중...
[V] 필터링 완료: 911건 (중복 제거됨)
[*] '/content/drive/Othercomputers/내 컴퓨터/새 폴더/양상추사진/양상추_테라웨이브/양상추 날짜/RGB_정면/1. 원본_핵심/251213-251226'로 파일 복사 시작 (병렬 작업)...


복사 진행률: 100%|██████████| 911/911 [01:08<00:00, 13.29it/s]


[★] 모든 작업이 완료되었습니다.
총 소요 시간: 0:01:19.899480
